In [1]:
import json
import os
import pandas as pd
import ast

In [2]:
# Get list of all files in the directory
files = os.listdir('./Filtered_Output/')
jsonl_files = [file for file in files if file.endswith('.jsonl') and file.startswith('multi-')]
print(jsonl_files)


['multi-dataset_gpt-4o-mini_0.0.jsonl', 'multi-dataset_gpt-4o-mini_0.2.jsonl', 'multi-dataset_gpt-4o-mini_0.6.jsonl', 'multi-dataset_gpt-4o-mini_0.4.jsonl', 'multi-dataset_gpt-4o-mini_1.0.jsonl', 'multi-dataset_gemini-2.5-flash_1.0.jsonl', 'multi-dataset_starcoder2_0.0.jsonl', 'multi-dataset_gemini-2.5-flash_0.4.jsonl', 'multi-dataset_Qwen_0.2.jsonl', 'multi-dataset_Qwen_0.0.jsonl', 'multi-dataset_gemini-2.5-flash_0.6.jsonl', 'multi-dataset_starcoder2_0.2.jsonl', 'multi-dataset_Qwen_1.0.jsonl', 'multi-dataset_starcoder2_0.6.jsonl', 'multi-dataset_Qwen_0.4.jsonl', 'multi-dataset_gemini-2.5-flash_0.2.jsonl', 'multi-dataset_gemini-2.5-flash_0.0.jsonl', 'multi-dataset_Qwen_0.6.jsonl', 'multi-dataset_starcoder2_1.0.jsonl', 'multi-dataset_starcoder2_0.4.jsonl', 'multi-dataset_starcoder2_0.8.jsonl', 'multi-dataset_Qwen_0.8.jsonl', 'multi-dataset_gemini-2.5-flash_0.8.jsonl', 'multi-dataset_gpt-4o-mini_0.8.jsonl']


In [3]:
def is_compilable(code):
    try:
        ast.parse(code)
    except SyntaxError:
        return False
    return True

In [4]:
def search(combined_data, current_cwe, file_name):
    direct_vulnerable = 0
    indirect_vulnerable = 0
    for line in combined_data:
        if file_name in line:
            target_cwe = line.split(',')[0]
            if 'cryptography' not in target_cwe:
                target_cwe = int(target_cwe)
            if current_cwe == target_cwe:
                direct_vulnerable += 1
            else:
                indirect_vulnerable += 1
    return direct_vulnerable, indirect_vulnerable


In [5]:
for file_name in jsonl_files:
    # if 'gpt-4' not in file_name:
    #     continue
    with open('./Filtered_Output/' + file_name, 'r') as f:
        data = [json.loads(line) for line in f.readlines()]

    print(len(data))
    full_name = '_'.join(file_name.split('.jsonl')[0].split('_'))
    temp = full_name.split('_')[-1]
    model_name = '_'.join(full_name.split('_')[:-1])
    print(model_name, temp)
    files = os.listdir(f'./CodeQL_Output/{full_name}')
    result_files = [file for file in files if file.endswith('.csv')]

    combined_data = []
    for result_file in result_files:
        target_cwe = result_file.split('.')[0].split('_')[-1]
        if '-' in target_cwe:
            target_cwe = target_cwe.split('-')[1]
        if '022bis' in target_cwe:
            target_cwe = '022'
        if 'cryptography'  not in target_cwe:
            target_cwe = int(target_cwe)

        with open(f'./CodeQL_Output/{full_name}/{result_file}', 'r') as f:
            lines = f.readlines()
        for line in lines:
            line = str(target_cwe)+','+model_name+','+temp+','+result_file+','+line.strip()
            combined_data.append(line)
    with open(f'./CWE_Results/{full_name}.csv', 'w') as f:
        f.write('\n'.join(combined_data)) 
    result = []
    for i in range(len(data)):
        id = data[i]['id']
        technique =  data[i]['technique']
        source = data[i]['source']
        language = data[i]['language']
        
        file_name = '_'.join(id.split('_')[2:])
        current_cwe = int(id.split('_')[-2:-1][0].split('cwe')[1])

        # print(id, current_cwe, file_name)

        for j in range(len(data[i]['output'])):
                current_file_name = file_name.replace('.py', f'_{j}_{language}.py')
                # print(current_file_name)
                direct_vulnerable, indirect_vulnerable = search(combined_data, current_cwe, current_file_name)
                # if direct_vulnerable>0:
                #     print(direct_vulnerable, indirect_vulnerable)
                is_compilable_flag = is_compilable(data[i]['output'][j]['cleared_code'])
                result.append([id,j, model_name, temp, technique, source, current_file_name, language, is_compilable_flag, direct_vulnerable, indirect_vulnerable])

        # break


    df = pd.DataFrame(result, columns=['id', 'index','model_name', 'Temp', 'technique', 'source', 'file_name','language','is_compilable', 'direct_vulnerable', 'indirect_vulnerable'])
    df.to_csv(f'./CodeQL_Results/{full_name}.csv', index=False)
    # break

2036
multi-dataset_gpt-4o-mini 0.0
2036
multi-dataset_gpt-4o-mini 0.2
2036
multi-dataset_gpt-4o-mini 0.6
2036
multi-dataset_gpt-4o-mini 0.4
2036
multi-dataset_gpt-4o-mini 1.0
2036
multi-dataset_gemini-2.5-flash 1.0
2036
multi-dataset_starcoder2 0.0
2036
multi-dataset_gemini-2.5-flash 0.4
2036
multi-dataset_Qwen 0.2
2036
multi-dataset_Qwen 0.0
2036
multi-dataset_gemini-2.5-flash 0.6
2036
multi-dataset_starcoder2 0.2
2036
multi-dataset_Qwen 1.0
2036
multi-dataset_starcoder2 0.6
2036
multi-dataset_Qwen 0.4
2036
multi-dataset_gemini-2.5-flash 0.2
2036
multi-dataset_gemini-2.5-flash 0.0
2036
multi-dataset_Qwen 0.6
2036
multi-dataset_starcoder2 1.0
2036
multi-dataset_starcoder2 0.4
2036
multi-dataset_starcoder2 0.8
2036
multi-dataset_Qwen 0.8
2036
multi-dataset_gemini-2.5-flash 0.8
2036
multi-dataset_gpt-4o-mini 0.8


In [6]:
# Get list of all files in the directory
files = os.listdir('./CodeQL_Results/')
csv_files = [file for file in files if file.endswith('.csv') and file.startswith('multi-')]
print(csv_files)

['multi-dataset_Qwen_0.0.csv', 'multi-dataset_gemini-2.5-flash_0.8.csv', 'multi-dataset_gpt-4o-mini_0.8.csv', 'multi-dataset_Qwen_0.2.csv', 'multi-dataset_Qwen_0.6.csv', 'multi-dataset_starcoder2_1.0.csv', 'multi-dataset_Qwen_0.4.csv', 'multi-dataset_starcoder2_0.4.csv', 'multi-dataset_Qwen_1.0.csv', 'multi-dataset_starcoder2_0.6.csv', 'multi-dataset_starcoder2_0.2.csv', 'multi-dataset_starcoder2_0.0.csv', 'multi-dataset_gpt-4o-mini_1.0.csv', 'multi-dataset_gemini-2.5-flash_1.0.csv', 'multi-dataset_starcoder2_0.8.csv', 'multi-dataset_gemini-2.5-flash_0.0.csv', 'multi-dataset_gpt-4o-mini_0.2.csv', 'multi-dataset_Qwen_0.8.csv', 'multi-dataset_gpt-4o-mini_0.0.csv', 'multi-dataset_gemini-2.5-flash_0.2.csv', 'multi-dataset_gemini-2.5-flash_0.6.csv', 'multi-dataset_gpt-4o-mini_0.4.csv', 'multi-dataset_gpt-4o-mini_0.6.csv', 'multi-dataset_gemini-2.5-flash_0.4.csv']


In [7]:
# combine all csv files
result = []
for file_name in csv_files:
    df = pd.read_csv('./CodeQL_Results/' + file_name)
    result.append(df)
df = pd.concat(result)
df.to_csv('./Combined_Multi-Results.csv', index=False)